<a href="https://colab.research.google.com/github/LaurenMitchell-tech/uvvisml/blob/main/notebooks/create_splits_song.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install .

import pandas as pd
import numpy as np
import torch
from lightning import pytorch as pl
from pathlib import Path
import pickle
import matplotlib.pyplot as plt

from chemprop import data, featurizers, models

from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

Cloning into 'chemprop'...
remote: Enumerating objects: 25619, done.
remote: Counting objects: 100% (381/381), done.
remote: Compressing objects: 100% (291/291), done.
remote: Total 25619 (delta 261), reused 90 (delta 90), pack-reused 25238 (from 3)
Receiving objects: 100% (25619/25619), 876.71 MiB | 20.65 MiB/s, done.
Resolving deltas: 100% (18353/18353), done.
Updating files: 100% (337/337), done.
/content/chemprop
Processing /content/chemprop
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

/content/chemprop/chemprop/data/collate.py:15: SyntaxWarning: invalid escape sequence '\s'
  """A :class:`BatchMolGraph` represents a batch of individual :class:`MolGraph`\s.
/content/chemprop/chemprop/data/collate.py:22: SyntaxWarning: invalid escape sequence '\s'
  """A list of individual :class:`MolGraph`\s to be batched together"""
/content/chemprop/chemprop/data/collate.py:65: SyntaxWarning: invalid escape sequence '\s'
  """the number of individual :class:`MolGraph`\s in this batch"""
/content/chemprop/chemprop/data/datasets.py:190: SyntaxWarning: invalid escape sequence '\s'
  """A :class:`MoleculeDataset` composed of :class:`MoleculeDatapoint`\s
/content/chemprop/chemprop/data/datasets.py:368: SyntaxWarning: invalid escape sequence '\s'
  """A :class:`CuikmolmakerDataset` composed of :class:`LazyMoleculeDatapoint`\s and a
/content/chemprop/chemprop/data/datasets.py:642: SyntaxWarning: invalid escape sequence '\s'
  """A :class:`ReactionDataset` composed of :class:`ReactionDatap

In [ ]:
os.chdir('/content')
!git clone https://github.com/LaurenMitchell-tech/uvvisml
os.chdir('/content/uvvisml/uvvisml/data')
from scaffold_splits import scaffold_split

Cloning into 'uvvisml'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 269 (delta 89), reused 64 (delta 34), pack-reused 123 (from 1)
Receiving objects: 100% (269/269), 16.60 MiB | 9.11 MiB/s, done.
Resolving deltas: 100% (134/134), done.
Updating files: 100% (69/69), done.


In [ ]:
def remove_invalid_smiles(df, smiles_col='smiles'):
    """Remove rows with invalid SMILES."""
    mask = df[smiles_col].apply(lambda x: Chem.MolFromSmiles(str(x)) is not None)
    return df.loc[mask].reset_index(drop=True)


def remove_nan_targets(df, target_col):
    """Remove rows where target is NaN."""
    return df.dropna(subset=[target_col]).reset_index(drop=True)


def basic_clean(df, smiles_col, target_col):
    """Standard cleaning used for all datasets."""
    df = df.copy()

    # normalize smiles column name
    if smiles_col != 'smiles':
        df = df.rename(columns={smiles_col: 'smiles'})

    # remove NaNs
    df = remove_nan_targets(df, target_col)

    # remove invalid molecules
    df = remove_invalid_smiles(df)

    return df.reset_index(drop=True)

def data_split_and_write(X, feature_names=None, target_names=['peakwavs_max'], solvation=False, split_type='scaffold',
                         scale_targets=False, write_files=False, random_seed=0):
    """Writes train, val, test CSV files for Chemprop with a given dataset.

    Parameters
    ----------
    X : pandas DataFrame
        DataFrame to be split (has columns: 'smiles', target_names, and feature_names
        (and 'solvent' if solvation=True))
    feature_names : list of str or None
        Names of feature columns in X to be added to feature files (default is None)
    target_names : list of str
        Names of target columns to be printed to files (default is ['peakwavs_max'])
    solvation : bool
        Specify whether to include solvents in target file (default is False)
    split_type : str
        which type of splitting to use ('scaffold', 'group_by_smiles', or 'random')
    scale_targets : bool
        whether to scale targets to have mean 0 and standard deviation 1 (default is False)
    write_files : bool
        whether to write the resulting splits to files
    random_seed : int or None
        number to provide for the seed / random_state arguments to make splits reproducible
        (use None if doing multiple splits for cross validation)

    """

    if split_type=='scaffold':
        X_train, X_val, X_test = scaffold_split(X, sizes=(0.8, 0.1, 0.1), balanced=True, seed=random_seed)
    elif split_type=='group_by_smiles': # Randomly split into train, val, and test sets such that no SMILES is in multiple sets
        gss1 = GroupShuffleSplit(n_splits=2, train_size=0.8, random_state=random_seed)
        train_idx, temp_idx = list(gss1.split(X, groups=X['smiles']))[0]
        X_train, X_temp = X.iloc[train_idx,:], X.iloc[temp_idx,:]
        gss2 = GroupShuffleSplit(n_splits=2, train_size=0.5, random_state=random_seed)
        val_idx, test_idx = list(gss2.split(X_temp, groups=X_temp['smiles']))[0]
        X_val, X_test = X_temp.iloc[val_idx, :], X_temp.iloc[test_idx, :]
    elif split_type=='random': # Randomly split into train, val, and test sets
        X_train = X.sample(frac=0.8, random_state=random_seed)
        X_temp = X.drop(X_train.index)
        X_val = X_temp.sample(frac=0.5, random_state=random_seed)
        X_test = X_temp.drop(X_val.index)

    if scale_targets:
        scaler = StandardScaler()
        scaler.fit(X_train[['target_names']])
        X_train[['target_names']] = scaler.transform(X_train[['target_names']])
        X_val[['target_names']] = scaler.transform(X_val[['target_names']])
        X_test[['target_names']] = scaler.transform(X_test[['target_names']])

    if write_files:
        # Name files
        train_target_file = 'smiles_target_train.csv'
        val_target_file = 'smiles_target_val.csv'
        test_target_file = 'smiles_target_test.csv'
        train_features_file = 'features_train.csv'
        val_features_file = 'features_val.csv'
        test_features_file = 'features_test.csv'

        # Write splits to CSVs
        if solvation:
            X_train[['smiles','solvent']+target_names].to_csv(train_target_file, index=False)
            X_val[['smiles','solvent']+target_names].to_csv(val_target_file, index=False)
            X_test[['smiles','solvent']+target_names].to_csv(test_target_file, index=False)
        else:
            X_train[['smiles']+target_names].to_csv(train_target_file, index=False)
            X_val[['smiles']+target_names].to_csv(val_target_file, index=False)
            X_test[['smiles']+target_names].to_csv(test_target_file, index=False)
        if feature_names:
            X_train[feature_names].to_csv(train_features_file, index=False)
            X_val[feature_names].to_csv(val_features_file, index=False)
            X_test[feature_names].to_csv(test_features_file, index=False)

    return X_train, X_val, X_test

In [ ]:
def handle_duplicates(df, target_col, cutoff=5, agg_source_col='multiple'):
    """
    Aggregates duplicate measurements in a DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns including 'smiles', 'solvent', target_col
    target_col : str
        Name of the target column to aggregate
    cutoff : float
        Maximum allowed standard deviation; rows with std > cutoff are dropped
    agg_source_col : str
        How to handle 'source' column: 'multiple' or 'random'

    Returns
    -------
    df : pd.DataFrame
        Cleaned DataFrame with duplicates aggregated
    """

    # Identify numeric columns other than the target
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != target_col]

    # Build aggregation dictionary
    agg_dict = {target_col: ['mean', 'std']}
    for col in numeric_cols:
        agg_dict[col] = 'mean'

    # Handle 'source' column if present
    if 'source' in df.columns:
        if agg_source_col == 'multiple':
            agg_dict['source'] = lambda x: 'multiple' if len(x) > 1 else x.iloc[0]
        elif agg_source_col == 'random':
            np.random.seed(0)
            agg_dict['source'] = lambda x: np.random.choice(x)

    # Group by smiles + solvent
    grouped = df.groupby(['smiles', 'solvent'], as_index=False).agg(agg_dict)

    # Drop rows with std above cutoff
    high_std_idx = grouped[grouped[target_col]['std'] > cutoff].index
    grouped = grouped.drop(index=high_std_idx)

    # Flatten MultiIndex columns
    grouped.columns = [c[0] if isinstance(c, tuple) else c for c in grouped.columns]

    return grouped

In [ ]:
# Set initial directory
DATA_DIR = os.getcwd()

# Load Song datasets
fluodb_path = r"/content/uvvisml/uvvisml/data/original/song/00_FluoDB.csv"
consolidation_path = r"/content/uvvisml/uvvisml/data/original/song/Dataset_Consolidation.csv"

fluodb_df = pd.read_csv(fluodb_path)
consolidation_df = pd.read_csv(consolidation_path, encoding="cp1252")


# -----------------------
# Clean FluoDB dataset
# -----------------------

# Standardize column names
consolidation_df = consolidation_df.rename(columns={
    "SMILES": "smiles",
    "Solvent": "solvent"
})

fluodb_df = basic_clean(
    fluodb_df,
    smiles_col='smiles',
    target_col='abs'
)

fluodb_df = fluodb_df.rename(columns={'abs':'peakwavs_max'})


# -----------------------
# Clean Consolidation dataset
# -----------------------

consolidation_df = basic_clean(
    consolidation_df,
    smiles_col='smiles',
    target_col='Ex (nm)'
)

consolidation_df = consolidation_df.rename(columns={
    'Ex (nm)': 'peakwavs_max'
})

In [ ]:
fluodb_df

,peakwavs_max,em,plqy,k,smiles,solvent,solvent_num,tag,tag_name,Molecular_Weight,LogP,TPSA,Avg_Gasteiger_Charge,Double_Bond_Count,Ring_Count,unimol_plus,split
0,439.0,460.0,0.6931,33830.0,CN(C)c1ccc(C#Cc2ccc3ccc4c(C#Cc5ccc(N(C)C)cc5)c...,CCCCCC,11,8,PAHs,488.634,7.51560,6.48,-0.041183,31,6,2.808881,train
1,344.0,508.0,0.0200,NaN,CC(=O)c1ccc2cc(C#Cc3cn([C@H]4C[C@H](O)[C@@H](C...,CC#N,4,8,PAHs,442.475,2.40980,123.49,-0.054583,22,5,3.439649,train
2,374.0,479.0,0.3800,23442.0,COc1ccccc1-c1nc(-c2ccccc2)c2ccccn12,CC#N,4,10,5p6,300.361,4.67690,26.53,-0.045242,22,4,3.873862,test
3,527.0,552.0,0.5400,41020.4,C#CC1=C(C)C2=C(C)c3c(C)c(C#C)c(C)n3[B-](F)(F)[...,CC(C)=O,12,5,BODIPY,310.156,3.49024,7.94,-0.030123,8,3,3.025253,train
4,391.0,531.0,0.1000,NaN,COC(=O)c1ccn2c(NC3CCCCC3)c(-c3ccc(OC)cc3O)nc2c1,CS(C)=O,7,10,5p6,395.459,4.24670,85.09,-0.055119,17,4,3.756094,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31333,432.0,563.0,0.0400,40970.0,CN(CCO)c1ccc(/C=C/C(=O)c2ccc(NC(=O)c3cc(F)cc(C...,CS(C)=O,7,13,6n6,486.465,5.42140,69.64,-0.044920,21,3,3.873625,test
31334,369.0,534.0,NaN,NaN,CCCCC1(CCCC)c2cc(C(=O)O)ccc2-c2ccc(N(c3ccccc3)...,CCO,2,6,Triphenylamine,489.659,9.50150,40.54,-0.047909,25,5,4.300098,train
31335,354.0,393.0,NaN,NaN,c1ccc(-c2cc(-c3ccc(N(c4ccccc4)c4ccccc4)cc3)cc(...,CCCCCC,11,6,Triphenylamine,474.607,9.55240,16.13,-0.044405,36,6,3.994269,test
31336,409.0,505.0,NaN,NaN,CCCCn1c2ccc(/C=C(\C#N)c3ccc([N+](=O)[O-])cc3)c...,c1ccccc1,15,3,Carbazole,746.915,12.53998,94.61,-0.036414,48,9,4.569871,valid


In [ ]:
consolidation_df

,smiles,solvent,peakwavs_max,Em (nm),ST (nm),QY,¦Å (cm-1M-1),Log(¦Å)
0,[BH3-][P+]1(c2ccccc2)c2ccccc2-c2sc3ccccc3c21,C1CCCCC1,324.0,395.0,71.0,NaN,NaN,NaN
1,[BH3-][P+]1(c2ccccc2)c2ccccc2-c2sc3ccccc3c21,CC#N,323.0,401.0,78.0,NaN,NaN,NaN
2,[BH3-][P+]1(c2ccccc2)c2ccccc2-c2sc3ccccc3c21,CCO,323.0,400.0,77.0,NaN,NaN,NaN
3,[BH3-][P+]1(c2ccccc2)c2ccccc2-c2sc3ccccc3c21,C(Cl)Cl,325.0,400.0,75.0,0.380,NaN,NaN
4,[C-][N+]/C(=C/c1ccc(N(c2ccc(Br)cc2)c2ccc(-c3nc...,C(Cl)(Cl)Cl,423.0,547.0,124.0,0.590,NaN,NaN
...,...,...,...,...,...,...,...,...
21702,S=C1CCCCC1,CCO,495.0,NaN,NaN,NaN,NaN,NaN
21703,S=P1(c2ccccc2)C(c2ccccc2)=Cc2cc3ccccc3cc21,C(Cl)Cl,387.0,425.0,38.0,0.001,NaN,NaN
21704,S=P1(c2ccccc2)c2cccc3c4c5ccccc5c5ccccc5c4c4ccc...,C(Cl)Cl,422.0,493.0,71.0,0.240,10471.0,4.02
21705,S=P1(c2ccccc2)c2cccc3ccc4ccc1n4c23,C(Cl)Cl,407.0,490.0,83.0,0.080,3162.0,3.50


In [ ]:
consolidation_df.to_csv("Dataset_Consolidation_cleaned.csv", index=False)

In [ ]:
fluodb_df.to_csv("00_FluoDB_cleaned.csv", index=False)

In [ ]:
# Optional duplicate aggregation
fluodb_df = handle_duplicates(fluodb_df, target_col='peakwavs_max')
consolidation_df = handle_duplicates(consolidation_df, target_col='peakwavs_max')

In [ ]:
for dataset_name, dataset_df in [
    ("fluodb", fluodb_df),
    ("consolidation", consolidation_df)
]:

    for split_type in ['random', 'group_by_smiles', 'scaffold']:

        output_dir = os.path.join(
            DATA_DIR,
            f"splits/{dataset_name}/{split_type}"
        )

        os.makedirs(output_dir, exist_ok=True)
        os.chdir(output_dir)

        _, _, _ = data_split_and_write(
            dataset_df,
            feature_names=None,
            target_names=['peakwavs_max'],
            solvation=False,
            split_type=split_type,
            scale_targets=False,
            write_files=True,
            random_seed=0
        )

100%|██████████| 21699/21699 [00:21<00:00, 992.85it/s] 
